# Setcover
## Жадный алгоритм

In [10]:
def read_instance(path):
    with open(path) as f:
        n, m = map(int, f.readline().split(' '))

        costs = []
        sets = []

        for _ in range(m):
            row = list(map(int, f.readline().split(' ')))
            costs.append(row[0])
            sets.append(set(row[1:]))

    return n, m, costs, sets


def greedy_set_cover(n, costs, sets):
    uncovered = set(range(1, n + 1))
    result = []

    while uncovered:
        best_score = float("inf")
        current_best = None

        for i, s in enumerate(sets):
            new_elements = uncovered & s
            if not new_elements:
                continue

            score = costs[i] / len(new_elements)

            if score < best_score:
                best_score = score
                current_best = i

        if current_best is None:
            break

        result.append(current_best)
        uncovered -= sets[current_best]

    return result

In [11]:
tests = ["data/sc_157_0", "data/sc_330_0", "data/sc_1000_11", "data/sc_5000_1", "data/sc_10000_5", "data/sc_10000_2"]
for test in tests:
    n, m, costs, sets = read_instance(test)
    solution = greedy_set_cover(n, costs, sets)

    total_cost = sum(costs[i] for i in solution)

    print(test, total_cost)

data/sc_157_0 98700
data/sc_330_0 31
data/sc_1000_11 170
data/sc_5000_1 36
data/sc_10000_5 76
data/sc_10000_2 192


Для оптимизации алгоритма можно попробовать добавить reverse-delete: постобработка результатов greedy, отсоритированное по убыванию стоимости, проверка на его необходимость (т.е меняется ли покрытие без него), в случае, если покрытие не изменяется, смело удаляем. Напишем кодом:

In [12]:
def reverse_delete(result, costs, sets, n):
    for i in sorted(result, key=lambda i: costs[i], reverse=True):
        others = [j for j in result if j != i]

        covered = set()
        for j in others:
            covered |= sets[j]

        if len(covered) == n:
            result.remove(i)

    return result

In [13]:
tests = ["data/sc_157_0", "data/sc_330_0", "data/sc_1000_11", "data/sc_5000_1", "data/sc_10000_5", "data/sc_10000_2"]
for test in tests:
    n, m, costs, sets = read_instance(test)
    solution = greedy_set_cover(n, costs, sets)
    solution = reverse_delete(solution, costs, sets, n)

    total_cost = sum(costs[i] for i in solution)

    print(test, total_cost)

data/sc_157_0 98700
data/sc_330_0 30
data/sc_1000_11 156
data/sc_5000_1 32
data/sc_10000_5 69
data/sc_10000_2 181


По времени почти не изменилось, зато дало буст почти на всех тестах -- 
| Тест         | Greedy | Greedy+ Reverse-delete |  Улучшение |
| ------------ | -----: | ---------------------: | ---------: |
| `sc_5000_1`  |     36 |                     32 | **11.11%** |
| `sc_10000_5` |     76 |                     69 |  **9.21%** |
| `sc_1000_11` |    170 |                    156 |  **8.24%** |
| `sc_10000_2` |    192 |                    181 |  **5.73%** |
| `sc_330_0`   |     31 |                     30 |  **3.23%** |
| `sc_157_0`   |  98700 |                  98700 |     **0%** |
